<a href="https://colab.research.google.com/github/Voidlindale/Voidlindale/blob/main/Reconstruction_plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install emcee corner -q
%matplotlib inline

# Configuration flags for cosmological probes
USE_SN             = True
USE_DESI_BAO       = True
USE_TRANSVERSE_BAO = False
USE_FS8            = True
USE_CMB            = True
USE_CC             = True

import os, numpy as np, pandas as pd, emcee, corner
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d
from scipy.stats import chi2, norm
import urllib.request, warnings

warnings.filterwarnings("ignore")

suffix = "THESIS_TRANSVERSE_TEST"
N_data = 0
if USE_SN:         suffix += "_SN";    N_data += 1590
if USE_DESI_BAO:       suffix += "_DESI";  N_data += 14
if USE_TRANSVERSE_BAO: suffix += "_TRBAO"
if USE_FS8:            suffix += "_FS8";   N_data += 7
if USE_CMB:            suffix += "_CMB";   N_data += 2
if USE_CC:             suffix += "_CC";    N_data += 34

print(f"🔧 ΣΕΝΑΡΙΟ: {suffix} | N_data αρχικό: {N_data}")

# Load observational datasets
if USE_SN:
    url_dat = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
    url_cov = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES_STAT%2BSYS.cov"
    if not os.path.exists("Pantheon+SH0ES.dat"): urllib.request.urlretrieve(url_dat, "Pantheon+SH0ES.dat")
    if not os.path.exists("Pantheon+SH0ES.cov"): urllib.request.urlretrieve(url_cov, "Pantheon+SH0ES.cov")
    df_sn = pd.read_csv("Pantheon+SH0ES.dat", sep=r'\s+')
    mask = df_sn['zHD'] > 0.01
    z_sn, mu_obs = df_sn[mask]['zHD'].values, df_sn[mask]['m_b_corr'].values
    with open("Pantheon+SH0ES.cov", 'r') as f:
        n_total = int(f.readline())
        cov_flat = np.loadtxt(f)
    inv_C_sn = np.linalg.inv(
        cov_flat.reshape((n_total, n_total))[
            np.ix_(np.where(mask.values)[0], np.where(mask.values)[0])
        ]
    )

if USE_DESI_BAO or USE_FS8:
    desi_z     = np.array([0.15, 0.51, 0.71, 0.93, 1.11, 1.32, 1.49])
    dm_rs_obs  = np.array([4.47, 13.62, 16.85, 21.71, 25.46, 27.81, 31.06])
    dh_rs_obs  = np.array([28.47, 20.98, 20.07, 16.09, 13.91, 13.14, 11.23])
    dm_rs_err  = np.array([0.15, 0.20, 0.22, 0.25, 0.30, 0.35, 0.45])
    dh_rs_err  = np.array([0.40, 0.45, 0.45, 0.50, 0.60, 0.70, 0.90])
    desi_r_corr = np.array([-0.40, -0.40, -0.40, -0.40, -0.40, -0.40, -0.40])
    fs8_obs    = np.array([0.444, 0.462, 0.451, 0.475, 0.482, 0.467, 0.432])
    fs8_err    = np.array([0.038, 0.040, 0.031, 0.025, 0.030, 0.045, 0.050])

if USE_TRANSVERSE_BAO:
    bao_tr_z   = np.array([0.38, 0.51, 0.61, 0.81, 1.52, 2.33])
    bao_tr_obs = np.array([10.23, 13.36, 15.45, 18.92, 26.69, 37.77])
    bao_tr_err = np.array([0.17, 0.21, 0.22, 0.51, 0.90, 2.13])
    N_data += len(bao_tr_z)

z_star, R_obs, R_err = 1089.92, 1.7502, 0.0046

cc_z = np.array([
    0.07, 0.09, 0.12, 0.17, 0.179, 0.199, 0.2, 0.27, 0.28, 0.35,
    0.352, 0.38, 0.4, 0.4004, 0.424, 0.44, 0.47, 0.4783, 0.48, 0.57,
    0.593, 0.68, 0.73, 0.781, 0.875, 0.88, 0.9, 1.037, 1.3, 1.363,
    1.43, 1.53, 1.75, 1.965
])
cc_h = np.array([
    69.0, 69.0, 68.5, 83.0, 75.0, 75.0, 72.9, 77.0, 88.8, 82.7,
    83.0, 83.0, 95.0, 77.0, 87.6, 82.6, 89.0, 80.9, 97.0, 92.4,
    104.0, 92.0, 97.3, 105.0, 125.0, 90.0, 117.0, 154.0, 168.0, 160.0,
    177.0, 140.0, 202.0, 186.5
])
cc_err = np.array([
    19.6, 12.0, 15.0, 8.0, 4.0, 5.0, 29.6, 14.0, 36.6, 8.4,
    14.0, 13.5, 17.0, 10.2, 7.8, 7.8, 49.6, 9.0, 62.0, 4.5,
    13.0, 8.0, 7.0, 12.0, 17.0, 40.0, 23.0, 20.0, 17.0, 33.6,
    18.0, 14.0, 40.0, 50.4
])

z_grid = np.concatenate([
    np.linspace(0, 5, 500, endpoint=False),
    np.linspace(5, z_star, 300)
])

def get_model_setup(mod):
    """Define parameter labels and initial guesses for each cosmological model."""
    if mod == 'LCDM':
        labels, p0 = [r"$\Omega_m$", r"$h$"], [0.31, 0.68]
    elif mod == 'wCDM':
        labels, p0 = [r"$\Omega_m$", r"$w_0$", r"$h$"], [0.31, -1.0, 0.68]
    elif mod == 'CPL':
        labels, p0 = [r"$\Omega_m$", r"$w_0$", r"$w_a$", r"$h$"], [0.31, -1.0, 0.0, 0.68]
    if USE_SN:  labels.append(r"$M$");         p0.append(-19.4)
    if USE_FS8: labels.append(r"$\sigma_8$"); p0.append(0.8)
    return len(p0), p0, labels

def E_z(z, Om, w0, wa, h):
    """Dimensionless Hubble expansion rate E(z) = H(z)/H0."""
    Or0  = 4.15e-5 / (h**2)
    f_de = ((1.0 + z)**(3.0*(1.0 + w0 + wa))) * np.exp(-3.0*wa*(z/(1.0 + z)))
    return np.sqrt(Or0*(1.0+z)**4.0 + Om*(1.0+z)**3.0 + (1.0-Om-Or0)*f_de)

def log_likelihood(theta, model):
    """Compute the total log-likelihood for the given model and parameters."""
    if model == 'LCDM':
        Om, h = theta[0], theta[1]
        w0, wa = -1.0, 0.0
        idx = 2
    elif model == 'wCDM':
        Om, w0, h = theta[0], theta[1], theta[2]
        wa = 0.0
        idx = 3
    elif model == 'CPL':
        Om, w0, wa, h = theta[0], theta[1], theta[2], theta[3]
        idx = 4

    if USE_SN:
        M = theta[idx]
        idx += 1
    else:
        M = -19.4

    if USE_FS8:
        s8 = theta[idx]
        idx += 1
    else:
        s8 = 0.8

    # Apply flat priors
    if not (0.2 < Om < 0.45 and 0.6 < h < 0.8):         return -np.inf
    if model != 'LCDM' and not (-1.8 < w0 < -0.2):        return -np.inf
    if model == 'CPL'  and not (-2.0 < wa  <  2.0):        return -np.inf
    if USE_SN  and not (-22.0 < M  < -17.0):                return -np.inf
    if USE_FS8 and not (  0.2 < s8 <    1.5):                return -np.inf

    chi2_val = 0.0
    rd_theo  = 147.09 * ((Om * h**2) / 0.143)**(-0.25)

    f_int = interp1d(
        z_grid,
        cumulative_trapezoid(1.0 / E_z(z_grid, Om, w0, wa, h), z_grid, initial=0),
        kind='linear', bounds_error=False, fill_value="extrapolate"
    )

    if USE_CMB:
        chi2_val += ((rd_theo - 147.09) / 0.26)**2
        chi2_val += ((np.sqrt(Om) * f_int(z_star) - R_obs) / R_err)**2

    if USE_SN:
        mu_theo   = 5*np.log10((299792.458/(100*h))*(1+z_sn)*f_int(z_sn)) + 25 + M
        chi2_val += (mu_obs - mu_theo).T @ inv_C_sn @ (mu_obs - mu_theo)

    if USE_DESI_BAO or USE_FS8:
        z_fs8_grid = np.linspace(0, 2.0, 200)
        Om_z_grid  = (Om*(1+z_fs8_grid)**3) / E_z(z_fs8_grid, Om, w0, wa, h)**2
        fs8_interp = interp1d(
            z_fs8_grid,
            cumulative_trapezoid(Om_z_grid**0.55 / (1+z_fs8_grid), z_fs8_grid, initial=0),
            kind='linear', bounds_error=False, fill_value="extrapolate"
        )
        for i, zi in enumerate(desi_z):
            Ez_i = E_z(zi, Om, w0, wa, h)
            if USE_DESI_BAO:
                dm       = (299792.458/(100*h)) * f_int(zi) / rd_theo
                dh       = (299792.458/(100*h*Ez_i)) / rd_theo
                diff_bao = np.array([dm - dm_rs_obs[i], dh - dh_rs_obs[i]])
                cov_bao  = np.array([
                    [dm_rs_err[i]**2,                    desi_r_corr[i]*dm_rs_err[i]*dh_rs_err[i]],
                    [desi_r_corr[i]*dm_rs_err[i]*dh_rs_err[i],    dh_rs_err[i]**2]
                ])
                chi2_val += diff_bao.T @ np.linalg.inv(cov_bao) @ diff_bao
            if USE_FS8:
                fs8_theo  = s8*(Om*(1+zi)**3/Ez_i**2)**0.55 * np.exp(-fs8_interp(zi))
                chi2_val += ((fs8_theo - fs8_obs[i]) / fs8_err[i])**2

    if USE_TRANSVERSE_BAO:
        for i, zi in enumerate(bao_tr_z):
            dm_theo   = (299792.458/(100*h)) * f_int(zi) / rd_theo
            chi2_val += ((dm_theo - bao_tr_obs[i]) / bao_tr_err[i])**2

    if USE_CC:
        chi2_val += np.sum(((100*h*E_z(cc_z, Om, w0, wa, h) - cc_h) / cc_err)**2)

    return -0.5 * chi2_val

# Run MCMC sampling for each model
n_steps, burn_in = 20000, 4000
N_walkers = 32

for mod in ['LCDM', 'wCDM', 'CPL']:
    ndim, p0, _ = get_model_setup(mod)
    print(f"\n🚀 Running MCMC for {mod}  (ndim={ndim}, walkers={N_walkers}, steps={n_steps})...")
    filename = f"mcmc_chain_{mod}_{suffix}.h5"
    backend  = emcee.backends.HDFBackend(filename)
    backend.reset(N_walkers, ndim)
    sampler  = emcee.EnsembleSampler(N_walkers, ndim, log_likelihood, args=[mod], backend=backend)
    sampler.run_mcmc(
        np.array(p0) + 1e-4 * np.random.randn(N_walkers, ndim),
        n_steps, progress=True
    )
    print(f"    ✔ Acceptance rate: {np.mean(sampler.acceptance_fraction)*100:.1f}%")

def gelman_rubin(chain):
    """Calculate the Gelman-Rubin convergence diagnostic R-hat."""
    N, M     = chain.shape[0], chain.shape[1]
    mean_w   = np.mean(chain, axis=0)
    mean_all = np.mean(mean_w, axis=0)
    B = (N/(M-1)) * np.sum((mean_w - mean_all)**2, axis=0)
    W = (1/(M*(N-1))) * np.sum((chain - mean_w)**2, axis=(0,1))
    return np.sqrt(((N-1)/N)*W + (1/N)*B) / np.sqrt(W)

def analyze_results():
    """Process chains, print parameter constraints, and generate diagnostic plots."""
    models_internal = ['LCDM', 'wCDM', 'CPL']
    display_names   = {'LCDM': r'$\Lambda$CDM', 'wCDM': r'$w$CDM', 'CPL': 'CPL'}
    colors          = {'LCDM': '#2ca02c',        'wCDM': '#1f77b4', 'CPL': '#d62728'}

    print("\n" + "="*95)
    print(f"🌟  FINAL ANALYSIS: {suffix}")
    print("="*95)

    chains, results_list = {}, []

    for mod in models_internal:
        _, _, labels = get_model_setup(mod)
        filename = f"mcmc_chain_{mod}_{suffix}.h5"
        reader   = emcee.backends.HDFBackend(filename, read_only=True)
        flat     = reader.get_chain(discard=burn_in, flat=True)
        raw      = reader.get_chain(discard=burn_in)
        chains[mod] = flat

        meds         = np.median(flat, axis=0)
        sig_up, sig_lo = np.percentile(flat, 84, axis=0) - meds, meds - np.percentile(flat, 16, axis=0)
        chi2_min       = -2 * log_likelihood(meds, mod)
        r_hat          = np.max(gelman_rubin(raw))

        results_list.append({'mod': mod, 'pretty_mod': display_names[mod], 'meds': meds})

        print(f"\n✅  {display_names[mod]}")
        print(f"    R-hat (max): {r_hat:.4f}{'  ✔' if r_hat < 1.01 else '  ⚠ non-converged'}   |   χ² min: {chi2_min:.2f}")
        for i in range(len(meds)):
            print(f"    {labels[i]:>12s}: {meds[i]:.4f}  +{sig_up[i]:.4f}  -{sig_lo[i]:.4f}")

        if mod == 'CPL':
            cov_w0_wa      = np.cov(flat[:, 1], flat[:, 2])
            diff           = np.array([-1.0 - np.mean(flat[:, 1]), 0.0 - np.mean(flat[:, 2])])
            delta_chi2_2d = diff.T @ np.linalg.inv(cov_w0_wa) @ diff
            sigma_2d      = norm.isf(chi2.sf(delta_chi2_2d, df=2) / 2)
            print(f"    Deviation from (-1,0): Δχ² = {delta_chi2_2d:.2f}  (~{sigma_2d:.2f}σ)")

        # Corner plot for individual models
        fig = corner.corner(
            flat, labels=labels, color=colors[mod],
            show_titles=True, title_fmt=".4f", quantiles=[0.16, 0.5, 0.84],
            levels=(0.68, 0.95), smooth=1.1, bins=40,
            fill_contours=True, plot_datapoints=False, alpha=0.6
        )
        fig.subplots_adjust(top=0.88)
        fig.suptitle(f"{display_names[mod]} Constraints — {suffix}", fontsize=14, y=0.98)
        plt.show()

    # Master overlay plot for cosmological parameters
    fig = corner.corner(
        chains['LCDM'][:, [0, 1]], labels=[r"$\Omega_m$", r"$h$"],
        color=colors['LCDM'], alpha=0.4, levels=(0.68, 0.95),
        smooth=1.1, fill_contours=True, plot_datapoints=False
    )
    corner.corner(chains['wCDM'][:, [0, 2]], fig=fig, color=colors['wCDM'],
                  alpha=0.4, levels=(0.68, 0.95), smooth=1.1,
                  fill_contours=True, plot_datapoints=False)
    corner.corner(chains['CPL'][:, [0, 3]], fig=fig, color=colors['CPL'],
                  alpha=0.4, levels=(0.68, 0.95), smooth=1.1,
                  fill_contours=True, plot_datapoints=False)
    plt.legend(
        handles=[mpatches.Patch(color=colors[m], label=display_names[m]) for m in models_internal],
        loc='upper right', bbox_to_anchor=(1.0, 1.0), fontsize=12
    )
    fig.suptitle("Master Overlay", fontsize=14, y=0.98)
    plt.show()

    # Om(z) diagnostic plot
    plt.figure(figsize=(9, 6))
    z_diag = np.linspace(0.01, 2.3, 150)
    for r in results_list:
        mod = r['mod']; meds = r['meds']
        if mod == 'LCDM':    w0_b, wa_b, h_b = -1.0,    0.0,     meds[1]
        elif mod == 'wCDM': w0_b, wa_b, h_b = meds[1], 0.0,     meds[2]
        elif mod == 'CPL':  w0_b, wa_b, h_b = meds[1], meds[2], meds[3]
        om_b   = meds[0]
        om_vals = [(E_z(zi, om_b, w0_b, wa_b, h_b)**2 - 1) / ((1+zi)**3 - 1) for zi in z_diag]
        plt.plot(z_diag, om_vals, label=r['pretty_mod'], lw=2.5, color=colors[mod], alpha=0.85)

    plt.axhline(results_list[0]['meds'][0], color='black', ls='--',
                label=r'ΛCDM reference $Ω_m$', alpha=0.6)
    plt.xlabel('Redshift  z', fontsize=13)
    plt.ylabel('Om(z)', fontsize=13)
    plt.title(r'Om(z) Diagnostic — Deviations from $\Lambda$CDM', fontsize=14)
    plt.legend(fontsize=11); plt.grid(alpha=0.2); plt.xlim(0.0, 2.3)
    plt.tight_layout(); plt.show()

    # Model comparison metrics (AIC, BIC)
    print("\n── Model Comparison ──────────────────────")
    print(f"{'Model':<10} {'k':>4} {'χ²_min':>10} {'AIC':>10} {'BIC':>10}")
    for r in results_list:
        k        = {'LCDM': 2, 'wCDM': 3, 'CPL': 4}[r['mod']]
        chi2_min = -2 * log_likelihood(r['meds'], r['mod'])
        aic      = chi2_min + 2 * k
        bic      = chi2_min + k * np.log(N_data)
        print(f"{r['pretty_mod']:<10} {k:>4} {chi2_min:>10.2f} {aic:>10.2f} {bic:>10.2f}")

analyze_results()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

# --- 1. Settings & Parameters ---
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12
})

Om0 = 0.3169
w0 = -0.9499
wa = 0.2083
z = np.linspace(0, 2.0, 500)

# --- 2. Calculations ---
f_z = (1 + z)**(3 * (1 + w0 + wa)) * np.exp(-3 * wa * z / (1 + z))
E_z = np.sqrt(Om0 * (1 + z)**3 + (1 - Om0) * f_z)
df_dz = 3 * f_z * ((1 + w0 + wa) / (1 + z) - wa / (1 + z)**2)
dE_dz = (1 / (2 * E_z)) * (3 * Om0 * (1 + z)**2 + (1 - Om0) * df_dz)

X_z = ((1 + z) / 3 * E_z * dE_dz) - 0.5 * Om0 * (1 + z)**3
V_z = E_z**2 - ((1 + z) / 3 * E_z * dE_dz) - 0.5 * Om0 * (1 + z)**3
w_z = w0 + wa * (z / (1 + z))

dphi_dz = (1 / (1 + z)) * np.sqrt(np.abs(2 * X_z)) / E_z
phi_z = cumulative_trapezoid(dphi_dz, z, initial=0)

# --- 3. Plots ---

# Plot 1: Equation of State w(z)
plt.figure(figsize=(8, 5))
plt.plot(z, w_z, color='darkred', lw=2.5, label=r'Reconstructed $w(z)$')
plt.axhline(-1, color='black', ls='--', label=r'$\Lambda$CDM')
plt.xlabel('Redshift $z$')
plt.ylabel('$w(z)$')
plt.title('Equation of State Evolution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Plot 2: Energy Decomposition X(z) and V(z)
plt.figure(figsize=(8, 5))
plt.plot(z, X_z, label=r'Kinetic $X(z)$', color='#E67E22', lw=2.5)
plt.plot(z, V_z, label=r'Potential $V(z)$', color='#0047AB', lw=2.5)
plt.axhline(0, color='black', lw=1.2)
plt.xlabel('Redshift $z$')
plt.ylabel(r'Energy Density $\rho / \rho_{crit,0}$')
plt.title('Field Energy Decomposition')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Plot 3: Potential Shape V(phi)
plt.figure(figsize=(8, 5))
plt.plot(phi_z, V_z, color='#4B0082', lw=3)
plt.xlabel(r'Field Displacement $\Delta \phi$ [$M_{pl}$]')
plt.ylabel(r'Potential $V(\phi)$')
plt.title('Potential Shape: Rolling Dynamics')
plt.grid(True, alpha=0.3)
plt.show()

# Plot 4: Stability Diagnostic X(z)
plt.figure(figsize=(8, 5))
plt.plot(z, X_z, color='#D35400', lw=2.5)
plt.axhline(0, color='black', ls=':', lw=1.5)
plt.fill_between(z, X_z, 0, color='#E67E22', alpha=0.2)
plt.xlabel('Redshift $z$')
plt.ylabel(r'Kinetic Energy $X(z)$')
plt.title('Stability Diagnostic: Kinetic Energy $X(z)$')
plt.grid(True, alpha=0.3)
plt.show()